In [0]:
from pyspark.sql import functions as F


In [0]:

%sql
Select * from `dataexpert-portfolio`.`market-data-tracker`.lb_watchlist_history
order by `_sort_by`

## RESTRICTIONS ON FREE VERSION RESTRICTED ME FROM USING SCD DBX CAPABILITIES
Due to this restriction I opted to use the merge into feature by just ranking the tables over partitioned symbols and emails. 

In [0]:
df_ranked = spark.sql('''
SELECT * ,
row_number() OVER(PARTITION BY symbol, email ORDER BY _timestamp DESC) as Row_Ct
FROM `dataexpert-portfolio`.`market-data-tracker`.`lb_watchlist_history`
ORDER BY symbol, row_ct
''')

df_ranked.show()

In [0]:
watchlist_filtered = df_ranked.filter((df_ranked.Row_Ct == 1))

if spark.catalog.tableExists("`dataexpert-portfolio`.`market-data-tracker`.slvr_watchlist_current"):
    existing_ids = spark.read.table("`dataexpert-portfolio`.`market-data-tracker`.slvr_watchlist_current").select("id")
    watchlist_filtered = watchlist_filtered.join(existing_ids, watchlist_filtered._pg_xid == existing_ids.id, "left_anti")
else:
    pass

watchlist_filtered.show()



In [0]:
newSchema = watchlist_filtered.withColumns({'date_updated': F.date_format('updated_at','yyyy-MM-dd'), 'time_updated': F.date_format('updated_at', 'HH:mm:ss'),'date_timestamped': F.date_format('_timestamp','yyyy-MM-dd'), 'time_stamped': F.date_format('_timestamp', 'HH:mm:ss')})

newSchema = newSchema.select('_pg_xid', "symbol", "latest_price", "date_updated", "time_updated", "date_timestamped", "time_stamped","_pg_change_type", "_sort_by", "email", "_pg_lsn")

newSchema = newSchema.withColumnsRenamed({'_pg_xid':'id', '_pg_change_type':'change_type', '_sort_by':'sort_by', '_pg_lsn':'lsn'})

newSchema = newSchema.withColumns({"tickerxuser_key": F.concat(F.col("symbol"), F.lit("_"), F.col("email")), "upload_timestamp": F.current_timestamp()})
newSchema.show()

watchlist_newSchema = newSchema

In [0]:
if spark.catalog.tableExists("`dataexpert-portfolio`.`market-data-tracker`.slvr_watchlist_current"):
    watchlist_newSchema.createOrReplaceTempView("source_updates")
    spark.sql('''
              MERGE INTO `dataexpert-portfolio`.`market-data-tracker`.slvr_watchlist_current AS target
              USING source_updates AS source
              ON target.tickerxuser_key = source.tickerxuser_key
              WHEN MATCHED THEN UPDATE SET *
              WHEN NOT MATCHED THEN INSERT *
              ''')
else:
    watchlist_newSchema.write.mode("overwrite").saveAsTable("`dataexpert-portfolio`.`market-data-tracker`.slvr_watchlist_current")

In [0]:
spark.read.table("`dataexpert-portfolio`.`market-data-tracker`.slvr_watchlist_current").show()

#spark.sql("DROP TABLE IF EXISTS `dataexpert-portfolio`.`market-data-tracker`.slvr_watchlist_current")